In [11]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from IPython.display import HTML, display 
from glob import glob
import ipywidgets as widgets
from IPython.display import clear_output

# Name of output directories
OUT_DIR = Path("drift_simple_report")
PLOTS_DIR = OUT_DIR / "plots"
assert PLOTS_DIR.exists(), "No se encontró drift_simple_report/plots. Ejecuta primero: python run_drift.py"
plt.rcParams.update({})  

### Functions

In [25]:
def Top5_table_graph(df_test: pd.DataFrame,
                     df_val: pd.DataFrame,
                     plots_dir: str = "drift_simple_report/plots",
                     img_test_name: str = "top5_test.png",
                     img_val_name: str = "top5_val.png"):
    """
    Muestra las tablas y gráficas Top-5 (Test y Val) lado a lado.
    Las tablas no tienen estilo, solo están centradas.
    Las gráficas se muestran centradas con títulos blancos.
    """

    # --- LIMPIEZA DE TABLAS ---
    for df in (df_test, df_val):
        for col in ("below_alpha", "drift_rate", "min_pvalue"):
            if col in df.columns:
                df.drop(columns=col, inplace=True)
        if "median_pvalue" in df.columns:
            df.rename(columns={"median_pvalue": "p_value"}, inplace=True)
        if "p_value" in df.columns:
            df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce").round(2)

    # --- TABLAS CENTRADAS Y LADO A LADO ---
    html_tables = f"""
    <div style="display:flex; justify-content:center; gap:50px; margin-top:10px;">
        <div style="text-align:center;">
            <h4>Top-5 Drift — Test</h4>
            {df_test.to_html(index=False)}
        </div>
        <div style="text-align:center;">
            <h4>Top-5 Drift — Val</h4>
            {df_val.to_html(index=False)}
        </div>
    </div>
    """
    display(HTML(html_tables))

    # --- IMÁGENES CENTRADAS ---
    plots_path = Path(plots_dir)
    test_img = plots_path / img_test_name
    val_img  = plots_path / img_val_name

    if test_img.exists() and val_img.exists():
        html_imgs = f"""
        <div style="display:flex; justify-content:center; gap:50px; margin-top:20px;">
            <div style="text-align:center;">
                <h4 style="color:white;">Top-5 Drift — Test</h4>
                <img src="{test_img.as_posix()}" width="560">
            </div>
            <div style="text-align:center;">
                <h4 style="color:white;">Top-5 Drift — Val</h4>
                <img src="{val_img.as_posix()}" width="560">
            </div>
        </div>
        """
        display(HTML(html_imgs))
    else:
        if not test_img.exists():
            print("No se encontró:", test_img)
        if not val_img.exists():
            print("No se encontró:", val_img)


def feature_image_viewer(plots_dir: str = "drift_simple_report/plots",
                         default_split: str = "test",
                         img_width: int = 900,      # ← ancho por defecto aumentado
                         img_height: int = 600):    # ← nuevo parámetro de alto
    """
    Visor interactivo con 2 imágenes FIJAS:
      - 1 histograma  {feature}_hist_all.png
      - 1 p-values    {feature}_pvalues_{split}.png
    """
    PLOTS_DIR = Path(plots_dir)

    # Descubrir features
    hist_files = glob(str(PLOTS_DIR / "*_hist_all.png"))
    features = sorted({Path(f).name.replace("_hist_all.png", "") for f in hist_files})
    if not features:
        print("No se encontraron histogramas en:", PLOTS_DIR)
        return

    # --- Widgets de control
    feat_dd  = widgets.Dropdown(options=features, description="Feature:", value=features[0])
    split_dd = widgets.Dropdown(options=[("test", "test"), ("val", "val")],
                                value=default_split, description="Split:")

    # --- Widgets de visualización
    title_html = widgets.HTML()
    hist_img   = widgets.Image(layout=widgets.Layout(width=f"{img_width}px", height=f"{img_height}px"))
    pval_img   = widgets.Image(layout=widgets.Layout(width=f"{img_width}px", height=f"{img_height}px"))
    hist_msg   = widgets.HTML("")
    pval_msg   = widgets.HTML("")

    def _read_binary(p: Path) -> bytes:
        with open(p, "rb") as f:
            return f.read()

    def update(*_):
        feature = feat_dd.value
        split   = split_dd.value

        title_html.value = f"<h4 style='margin:4px 0'>Feature: <code>{feature}</code> &nbsp;|&nbsp; Split: <code>{split}</code></h4>"

        # Rutas
        hist_path = PLOTS_DIR / f"{feature}_hist_all.png"
        pval_path = PLOTS_DIR / f"{feature}_pvalues_{split}.png"

        # Histograma
        if hist_path.exists():
            hist_img.value = _read_binary(hist_path)
            hist_img.format = 'png'
            hist_msg.value = ""
        else:
            hist_img.value = b""
            hist_msg.value = f"<span style='color:#b00'>No se encontró histograma: {hist_path.name}</span>"

        # p-values
        if pval_path.exists():
            pval_img.value = _read_binary(pval_path)
            pval_img.format = 'png'
            pval_msg.value = ""
        else:
            pval_img.value = b""
            pval_msg.value = f"<span style='color:#b00'>No se encontró p-values: {pval_path.name}</span>"

    # Conectar cambios
    feat_dd.observe(update, names="value")
    split_dd.observe(update, names="value")

    # Layout final
    controls = widgets.HBox([feat_dd, split_dd])
    hist_box = widgets.VBox([widgets.HTML("<b>Histograma</b>"), hist_img, hist_msg])
    pval_box = widgets.VBox([widgets.HTML("<b>p-values</b>"), pval_img, pval_msg])
    ui = widgets.VBox([controls, title_html, hist_box, pval_box])

    display(ui)
    update()  # render inicial



### Top 5 most dfrifted features


In [8]:
top5_test = pd.read_csv("drift_simple_report/top5_test.csv")
top5_val = pd.read_csv("drift_simple_report/top5_val.csv")
Top5_table_graph(top5_test, top5_val, "drift_simple_report/plots", "top5_test.png", "top5_val.png")

Feature,windows,p_value
ADI,11,0.0
OBV,11,0.0
VPT,11,0.0
ATR_14,11,0.0
ROC_45,11,0.0
Feature,windows,p_value
ADI,11,0.0
VPT,11,0.0
OBV,11,0.0
ATR_14,11,0.0


In [26]:
feature_image_viewer(plots_dir="drift_simple_report/plots", default_split="val", img_width=600)
